<a href="https://colab.research.google.com/github/ANEESAAYAZ/Internship_RidgeandLassoRegression/blob/main/Internship_RidgeandLassoRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
pd.set_option("display.width",200)

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv( "properties_features.csv", parse_dates=["listed_date"])
model_df = df[df["price_aed"].notna()].copy()

FORBIDDEN = [ "price_per_sqft", "rental_yield_pct",
              "log_price_aed", "area_price_encoded", "days_on_market",
              "sold_fast", "listing_status", "annual_rent_aed",
              "log_annual_rent_aed"]
DROP = FORBIDDEN + [ "listed_date", "description", "price_aed",
                    "latitude", "longitude"]

X = model_df.drop( columns=[c for c in DROP if c in model_df.columns])
y = np.log1p(model_df["price_aed"])

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

def make_preprocessor():
  return ColumnTransformer([
      ("num", Pipeline([ ("i",SimpleImputer(strategy="median")),
                          ("s", StandardScaler())]), numeric_cols),
      ("cat", Pipeline([("i",SimpleImputer(strategy="most_frequent")),
                        ("e", OneHotEncoder( handle_unknown="ignore",
                                            sparse_output=False,
                                             min_frequency=10 )) ]), categorical_cols),
  ])
X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2, random_state=42)

n_feat = make_preprocessor().fit(X_train).transform(X_train).shape[1]

print(f"\ntrain {len(X_train)} rows, {n_feat} features after encoding")
print(f"ratio: {n_feat / len(X_train):.3f} features per row")


train 2400 rows, 188 features after encoding
ratio: 0.078 features per row


**Ridge Regression**

In [3]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

unscaled = ColumnTransformer([
          ("num", SimpleImputer(strategy="median"), numeric_cols),
          ("cat", Pipeline([ ("i", SimpleImputer(strategy="most_frequent")),
                              ("e", OneHotEncoder(handle_unknown="ignore",
                                                  sparse_output=False,
                                                  min_frequency=10  )) ]), categorical_cols),])

print("\n--- the same Ridge, with and without scaling ---")
print(f"  {'alpha':>8s} {'scaled R2':>11s} {'unscaled R2':>12s}")

for a in [1, 10, 100, 1000, 10000]:
  s = Pipeline([ ("p", make_preprocessor()), ("m",Ridge(alpha=a)) ]).fit(X_train, y_train)
  u = Pipeline([ ("p", unscaled), ("m", Ridge(alpha=a))]).fit(X_train, y_train)
  print(f" {a:8g} {r2_score(y_test, s.predict(X_test)):11.4f} "
        f"{r2_score(y_test, u.predict(X_test)):12.4f}")

scaled_pipe = Pipeline([ ("p", make_preprocessor()), ("m",
                        Ridge(alpha=10.0))]).fit(X_train, y_train)
unscaled_pipe = Pipeline([ ("p", unscaled), ("m",
                          Ridge(alpha=10.0))]).fit(X_train, y_train)

sc = pd.Series( scaled_pipe.named_steps["m"].coef_,index=scaled_pipe.named_steps["p"].get_feature_names_out())

un = pd.Series( unscaled_pipe.named_steps["m"].coef_,index=unscaled_pipe.named_steps["p"].get_feature_names_out())

print("\n  at alpha=10, coefficient on built_up_area_sqft (runs to 17,000):")
print(f"  scaled  {sc['num__built_up_area_sqft']:+.6f}")
print(f"  unscaled {un['num__built_up_area_sqft']:+.6f}")
print("  and on maids_room (a 0/1 flag):")
print(f"  scaled  {sc['num__maids_room']:+.6f}")
print(f"  unscaled {un['num__maids_room']:+.6f}")


--- the same Ridge, with and without scaling ---
     alpha   scaled R2  unscaled R2
        1      0.9446       0.9442
       10      0.9334       0.9336
      100      0.8953       0.8957
     1000      0.8193       0.8246
    10000      0.6051       0.7005

  at alpha=10, coefficient on built_up_area_sqft (runs to 17,000):
  scaled  +0.089733
  unscaled +0.000054
  and on maids_room (a 0/1 flag):
  scaled  +0.055568
  unscaled +0.125994


In [6]:
alphas = [0.0001, 0.01, 0.1, 1, 10, 100, 1000, 10000]
print("\n--- Ridge: raising alpha ---\n")
print(f"{'alpha':>10s} {'train R2':>10s} {'test R2':>10s} {'largest |b|':>13s} {'sum |b|':>10s}")
ridge_path = []
for a in alphas:
  p = Pipeline([ ("p", make_preprocessor()), ("m", Ridge(alpha=a)) ]).fit(X_train, y_train)
  c = p.named_steps["m"].coef_
  row = (a, r2_score(y_train, p.predict(X_train)), r2_score(y_test, p.predict(X_test)),
         np.abs(c).max(),np.abs(c).sum())
  ridge_path.append(row)
  print(f"  {a:10g} {row[1]:10.4f} {row[2]:10.4f} {row[3]:13.3f} {row[4]:10.1f}")


--- Ridge: raising alpha ---

     alpha   train R2    test R2   largest |b|    sum |b|
      0.0001     0.9431     0.9456         5.580       68.0
        0.01     0.9430     0.9460         2.031       40.3
         0.1     0.9428     0.9461         0.912       31.3
           1     0.9421     0.9446         0.828       24.3
          10     0.9345     0.9334         0.547       16.3
         100     0.8991     0.8953         0.329        7.6
        1000     0.8157     0.8193         0.176        3.0
       10000     0.5996     0.6051         0.076        1.1


**Ordinary Linear Regression:**



In [8]:
from sklearn.linear_model import LinearRegression

GEO = ["km_to_burj_khalifa", "km_to_dxb_airport",
        "km_to_dubai_marina", "km_to_abu_dhabi_corniche",
        "distance_to_downtown_km"]
ols = Pipeline([ ("p", make_preprocessor()), ("m",LinearRegression())]).fit(X_train, y_train)
rdg = Pipeline([ ("p", make_preprocessor()), ("m",Ridge(alpha=10.0))]).fit(X_train, y_train)

names = ols.named_steps["p"].get_feature_names_out()
c_ols = pd.Series(ols.named_steps["m"].coef_, index=names)
c_rdg = pd.Series(rdg.named_steps["m"].coef_, index=names)

print("\n--- the collinear geo block, before and after a penalty ---")
print(f"  {'feature':32s} {'OLS':>10s} {'Ridge(10)':>10s}")

for g in GEO:
  print(f" {g:32s} {c_ols['num__' + g]:10.3f} {c_rdg['num__' + g]:10.3f}")
print(f"\n  largest |coefficient| overall: OLS {np.abs(c_ols).max():.2f}"
      f" Ridge {np.abs(c_rdg).max():.2f}")


--- the collinear geo block, before and after a penalty ---
  feature                                 OLS  Ridge(10)
 km_to_burj_khalifa                   -6.073     -0.101
 km_to_dxb_airport                     4.780      0.253
 km_to_dubai_marina                    1.270     -0.002
 km_to_abu_dhabi_corniche             -1.643     -0.000
 distance_to_downtown_km               0.344     -0.086

  largest |coefficient| overall: OLS 6.07 Ridge 0.55


**Lasso Regression**

In [9]:
from sklearn.linear_model import Lasso

print("\n--- Lasso: how many features survive? ---")
print(f"  {'alpha':>10s} {'test R2':>10s} {'non-zero':>10s} {'dropped':>9s}")

for a in [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]:
  p = Pipeline([ ("p", make_preprocessor()),
                  ("m", Lasso(alpha=a, max_iter=20000))]).fit(X_train, y_train)
  c = p.named_steps["m"].coef_
  nz = int((c != 0).sum())
  print(f" {a:10g} {r2_score(y_test, p.predict(X_test)):10.4f} "
        f"{nz:10d} {len(c) - nz:9d}")



--- Lasso: how many features survive? ---
       alpha    test R2   non-zero   dropped
     0.0001     0.9446        152        36
      0.001     0.9284         94        94
      0.005     0.8928         44       144
       0.01     0.8759         34       154
       0.05     0.7768         15       173
        0.1     0.6986          7       181


**ElasticNet Regression**

In [10]:
from sklearn.linear_model import ElasticNet

print("\n--- ElasticNet: l1_ratio slides from Ridge to Lasso ---")
print(f"{'l1_ratio':>10s} {'test R2':>10s} {'non-zero':>10s} behaviour")

for r in [0.0, 0.15, 0.5, 0.85, 1.0]:
  p = Pipeline([ ("p", make_preprocessor()),
                ("m", ElasticNet(alpha=0.005, l1_ratio=max(r, 1e-4),
                                max_iter=20000))]).fit(X_train, y_train)
  c = p.named_steps["m"].coef_
  label = "pure Ridge" if r == 0.0 else "pure Lasso" if r == 1.0 else "blend"
  print(f" {r:10.2f} {r2_score(y_test, p.predict(X_test)):10.4f} "
        f"{int((c != 0).sum()):10d}  {label}")


--- ElasticNet: l1_ratio slides from Ridge to Lasso ---
  l1_ratio    test R2   non-zero behaviour
       0.00     0.9314        187  pure Ridge
       0.15     0.9226        119  blend
       0.50     0.9052         66  blend
       0.85     0.8959         48  blend
       1.00     0.8928         44  pure Lasso
